In [1]:
import pygame

pygame 2.6.1 (SDL 2.28.4, Python 3.13.15)
Hello from the pygame community. https://www.pygame.org/contribute.html


# 1. Минимальное окно и главный цикл

Интерактивное приложение не выполняется один раз сверху вниз.  
После создания окна оно многократно повторяет один и тот же цикл:

```text
события → ввод → изменение состояния → отрисовка → следующий кадр
```


In [2]:
def demo_1_window():
    pygame.init()

    width, height = 800, 500
    screen = pygame.display.set_mode((width, height))
    pygame.display.set_caption("PyGame demo 1")

    clock = pygame.time.Clock()
    running = True

    try:
        while running:
            clock.tick(60)

            # EVENTS
            for event in pygame.event.get():
                if event.type == pygame.QUIT:
                    running = False

                elif event.type == pygame.KEYDOWN:
                    if event.key == pygame.K_ESCAPE:
                        running = False

            # RENDER
            screen.fill((245, 245, 245))

            pygame.draw.rect(
                screen,
                (90, 130, 210),
                pygame.Rect(280, 170, 240, 120),
            )

            # PRESENT
            pygame.display.flip()

    finally:
        pygame.quit()


demo_1_window()


# 2. Непрерывное движение и `delta time`

Для движения важно различать:

> **Событие**: `pygame.KEYDOWN`
> Подходит для действий, которые происходят один раз на нажатие.

> **Состояние клавиатуры**: `pygame.key.get_pressed()`
> Подходит для непрерывного движения при удержании клавиши.


In [3]:
def demo_2_movement():
    pygame.init()

    width, height = 900, 550
    screen = pygame.display.set_mode((width, height))
    pygame.display.set_caption("PyGame demo 2 — movement")

    clock = pygame.time.Clock()

    position = pygame.Vector2(width / 2, height / 2)
    speed = 240.0
    radius = 24

    colors = [
        (70, 130, 220),
        (60, 180, 110),
        (220, 90, 80),
        (230, 175, 60),
    ]
    color_index = 0

    running = True

    try:
        while running:
            dt = clock.tick(60) / 1000.0

            # EVENTS: разовые действия
            for event in pygame.event.get():
                if event.type == pygame.QUIT:
                    running = False

                elif event.type == pygame.KEYDOWN:
                    if event.key == pygame.K_ESCAPE:
                        running = False

                    elif event.key == pygame.K_SPACE:
                        color_index = (color_index + 1) % len(colors)

            # INPUT: непрерывное управление
            keys = pygame.key.get_pressed()

            direction = pygame.Vector2(0, 0)

            if keys[pygame.K_w]:
                direction.y -= 1
            if keys[pygame.K_s]:
                direction.y += 1
            if keys[pygame.K_a]:
                direction.x -= 1
            if keys[pygame.K_d]:
                direction.x += 1

            if direction.length_squared() > 0:
                direction = direction.normalize()

            # UPDATE
            position += direction * speed * dt

            position.x = max(radius, min(width - radius, position.x))
            position.y = max(radius, min(height - radius, position.y))

            # RENDER
            screen.fill((248, 248, 248))

            pygame.draw.circle(
                screen,
                colors[color_index],
                (round(position.x), round(position.y)),
                radius,
            )

            pygame.display.flip()

    finally:
        pygame.quit()


# WASD — движение
# SPACE — смена цвета
# ESC — выход
demo_2_movement()


### Главное выражение

```python
position += direction * speed * dt
```

Здесь:

- `direction` — направление;
- `speed` — скорость в пикселях в секунду;
- `dt` — длительность предыдущего кадра в секундах.

Так движение зависит от времени, а не от числа кадров.

Нормализация:

```python
direction = direction.normalize()
```

нужна для того, чтобы движение по диагонали не было быстрее движения по одной оси.


# 3. `pygame.Rect`: препятствия и зона взаимодействия

Для практической работы студентам понадобится уметь:

- описывать области сцены;
- проверять пересечения;
- не пропускать объект через препятствие;
- определять, находится ли объект в зоне взаимодействия.

Для этого достаточно `pygame.Rect`.


In [4]:
def demo_3_rects_and_interaction():
    pygame.init()

    width, height = 1000, 600
    screen = pygame.display.set_mode((width, height))
    pygame.display.set_caption("PyGame demo 3 — Rect and interaction")

    clock = pygame.time.Clock()
    font = pygame.font.Font(None, 30)
    small_font = pygame.font.Font(None, 24)

    position = pygame.Vector2(120, 300)
    speed = 260.0
    radius = 22

    # Непроходимое препятствие
    obstacle_rect = pygame.Rect(420, 180, 180, 240)

    # Невидимая логическая зона рядом с препятствием
    interaction_rect = pygame.Rect(620, 230, 180, 140)

    message = "Подойдите к зоне взаимодействия"

    running = True

    try:
        while running:
            dt = clock.tick(60) / 1000.0

            # EVENTS
            for event in pygame.event.get():
                if event.type == pygame.QUIT:
                    running = False

                elif event.type == pygame.KEYDOWN:
                    if event.key == pygame.K_ESCAPE:
                        running = False

                    elif event.key == pygame.K_e:
                        object_rect = pygame.Rect(0, 0, radius * 2, radius * 2)
                        object_rect.center = (
                            round(position.x),
                            round(position.y),
                        )

                        if object_rect.colliderect(interaction_rect):
                            message = "Действие выполнено"
                        else:
                            message = "Объект находится слишком далеко"

            # INPUT
            keys = pygame.key.get_pressed()

            direction = pygame.Vector2(0, 0)

            if keys[pygame.K_w]:
                direction.y -= 1
            if keys[pygame.K_s]:
                direction.y += 1
            if keys[pygame.K_a]:
                direction.x -= 1
            if keys[pygame.K_d]:
                direction.x += 1

            if direction.length_squared() > 0:
                direction = direction.normalize()

            # UPDATE: предполагаемая новая позиция
            next_position = position + direction * speed * dt

            next_position.x = max(radius, min(width - radius, next_position.x))
            next_position.y = max(radius, min(height - radius, next_position.y))

            next_rect = pygame.Rect(0, 0, radius * 2, radius * 2)
            next_rect.center = (
                round(next_position.x),
                round(next_position.y),
            )

            # Если столкновения нет — принимаем новую позицию
            if not next_rect.colliderect(obstacle_rect):
                position = next_position

            object_rect = pygame.Rect(0, 0, radius * 2, radius * 2)
            object_rect.center = (
                round(position.x),
                round(position.y),
            )

            can_interact = object_rect.colliderect(interaction_rect)

            # RENDER
            screen.fill((246, 246, 246))

            pygame.draw.rect(
                screen,
                (100, 110, 125),
                obstacle_rect,
                border_radius=10,
            )

            # Для учебной демонстрации зону взаимодействия показываем рамкой
            pygame.draw.rect(
                screen,
                (60, 170, 90) if can_interact else (175, 175, 175),
                interaction_rect,
                width=3,
                border_radius=10,
            )

            pygame.draw.circle(
                screen,
                (70, 130, 220),
                (round(position.x), round(position.y)),
                radius,
            )

            if can_interact:
                prompt = font.render(
                    "E — выполнить действие",
                    True,
                    (40, 120, 65),
                )
                screen.blit(prompt, (620, 190))

            msg = font.render(message, True, (40, 40, 40))
            screen.blit(msg, (25, 25))

            info = small_font.render(
                f"FPS: {clock.get_fps():.0f} | WASD — движение | E — действие | ESC — выход",
                True,
                (70, 70, 70),
            )
            screen.blit(info, (25, 565))

            pygame.display.flip()

    finally:
        pygame.quit()


demo_3_rects_and_interaction()


### Что здесь важно

Для препятствия:

```python
next_rect.colliderect(obstacle_rect)
```

проверяет пересечение предполагаемой новой позиции с прямоугольной областью.

Для зоны взаимодействия:

```python
object_rect.colliderect(interaction_rect)
```

позволяет определить, доступно ли некоторое действие.

Это уже даёт студенту общий механизм:

```text
объект подошёл к области
        ↓
условие стало True
        ↓
появилась подсказка
        ↓
пользователь нажал E
        ↓
изменилось состояние программы
```


# 4. Простое состояние и таймер без `time.sleep()`

В практической работе потребуется состояние, которое меняется со временем.

Для этого не нужно останавливать приложение.

Неправильно:

```python
time.sleep(5)
```

Пока выполняется `sleep`, окно не обновляется и не обрабатывает события.

Правильный принцип — хранить оставшееся время и уменьшать его на каждом кадре.


In [2]:
def demo_4_state_and_timer():
    pygame.init()

    width, height = 800, 420
    screen = pygame.display.set_mode((width, height))
    pygame.display.set_caption("PyGame demo 4 — state and timer")

    clock = pygame.time.Clock()
    font = pygame.font.Font(None, 34)
    small_font = pygame.font.Font(None, 26)

    state = "IDLE"
    duration = 5.0
    time_left = 0.0

    running = True

    try:
        while running:
            dt = clock.tick(60) / 1000.0

            # EVENTS
            for event in pygame.event.get():
                if event.type == pygame.QUIT:
                    running = False

                elif event.type == pygame.KEYDOWN:
                    if event.key == pygame.K_ESCAPE:
                        running = False

                    elif event.key == pygame.K_SPACE and state == "IDLE":
                        state = "RUNNING"
                        time_left = duration

                    elif event.key == pygame.K_r and state == "DONE":
                        state = "IDLE"
                        time_left = 0.0

            # UPDATE
            if state == "RUNNING":
                time_left -= dt

                if time_left <= 0:
                    time_left = 0
                    state = "DONE"

            # RENDER
            screen.fill((248, 248, 248))

            if state == "IDLE":
                text = "SPACE — запустить таймер"

            elif state == "RUNNING":
                text = f"Выполнение: {time_left:.1f} c"

            else:
                text = "Готово. R — сброс"

            label = font.render(text, True, (35, 35, 35))
            screen.blit(label, (40, 55))

            # Полоса прогресса
            bar_x, bar_y = 40, 150
            bar_width, bar_height = 700, 32

            pygame.draw.rect(
                screen,
                (215, 215, 215),
                (bar_x, bar_y, bar_width, bar_height),
                border_radius=6,
            )

            if state == "IDLE":
                progress = 0.0
            elif state == "RUNNING":
                progress = 1.0 - time_left / duration
            else:
                progress = 1.0

            filled_width = int(bar_width * progress)

            pygame.draw.rect(
                screen,
                (70, 160, 100),
                (bar_x, bar_y, filled_width, bar_height),
                border_radius=6,
            )

            state_label = small_font.render(
                f"State: {state}",
                True,
                (70, 70, 70),
            )
            screen.blit(state_label, (40, 230))

            fps_label = small_font.render(
                f"FPS: {clock.get_fps():.0f}",
                True,
                (70, 70, 70),
            )
            screen.blit(fps_label, (40, 270))

            pygame.display.flip()

    finally:
        pygame.quit()


# SPACE — запустить
# R — сбросить после завершения
# ESC — выйти
demo_4_state_and_timer()


# 5. Минимальная архитектура, которую должны сохранить студенты

Даже если всё приложение пока находится в одном `.py`-файле, полезно разделять четыре этапа:

```text
EVENTS
  ↓
INPUT
  ↓
UPDATE
  ↓
RENDER
```

Концептуально:

```python
while running:
    dt = clock.tick(60) / 1000.0

    handle_events()
    handle_input()
    update(dt)
    render()
```

### Events

Однократные действия:

- `Esc`;
- `Space`;
- `E`;
- выбор пункта меню.

### Input

Непрерывное состояние клавиш:

- `W`;
- `A`;
- `S`;
- `D`.

### Update

Изменение состояния:

- координаты;
- столкновения;
- таймеры;
- текущее состояние;
- числовые параметры.

### Render

Только отображение:

- фон;
- прямоугольники;
- круги;
- текст;
- индикаторы.

Функция отрисовки не должна неожиданно менять состояние приложения.


In [8]:
def demo_with_functions():
    pygame.init()

    WIDTH, HEIGHT, FPS = 900, 550, 60

    screen = pygame.display.set_mode((WIDTH, HEIGHT))
    pygame.display.set_caption("PyGame — разделение по функциям")

    clock = pygame.time.Clock()
    font = pygame.font.Font(None, 28)

    # -------------------------------------------------
    # Состояние приложения
    # -------------------------------------------------
    state = {
        "running": True,
        "position": pygame.Vector2(WIDTH / 2, HEIGHT / 2),
        "speed": 240.0,
        "radius": 24,
        "color_index": 0,
        "colors": [
            (70, 130, 220),
            (60, 180, 110),
            (220, 90, 80),
            (230, 175, 60),
        ],
    }

    obstacle = pygame.Rect(610, 180, 170, 190)

    # -------------------------------------------------
    # 1. Обработка событий
    # -------------------------------------------------
    def handle_events():
        for event in pygame.event.get():

            if event.type == pygame.QUIT:
                state["running"] = False

            elif event.type == pygame.KEYDOWN:

                if event.key == pygame.K_ESCAPE:
                    state["running"] = False

                elif event.key == pygame.K_SPACE:
                    state["color_index"] = (
                        state["color_index"] + 1
                    ) % len(state["colors"])

    # -------------------------------------------------
    # 2. Непрерывный ввод
    # -------------------------------------------------
    def handle_input():
        keys = pygame.key.get_pressed()

        direction = pygame.Vector2(0, 0)

        if keys[pygame.K_w]:
            direction.y -= 1

        if keys[pygame.K_s]:
            direction.y += 1

        if keys[pygame.K_a]:
            direction.x -= 1

        if keys[pygame.K_d]:
            direction.x += 1

        if direction.length_squared() > 0:
            direction = direction.normalize()

        return direction

    # -------------------------------------------------
    # 3. Изменение состояния
    # -------------------------------------------------
    def update(direction, dt):
        next_position = (
            state["position"]
            + direction * state["speed"] * dt
        )

        radius = state["radius"]

        # Ограничение границами окна
        next_position.x = max(
            radius,
            min(WIDTH - radius, next_position.x),
        )

        next_position.y = max(
            radius,
            min(HEIGHT - radius, next_position.y),
        )

        # Проверяем предполагаемую новую позицию
        next_rect = pygame.Rect(
            0,
            0,
            radius * 2,
            radius * 2,
        )

        next_rect.center = (
            round(next_position.x),
            round(next_position.y),
        )

        # Применяем движение только если нет столкновения
        if not next_rect.colliderect(obstacle):
            state["position"] = next_position

    # -------------------------------------------------
    # 4. Отрисовка
    # -------------------------------------------------
    def render():
        screen.fill((248, 248, 248))

        pygame.draw.rect(
            screen,
            (105, 115, 130),
            obstacle,
            border_radius=10,
        )

        pygame.draw.circle(
            screen,
            state["colors"][state["color_index"]],
            (
                round(state["position"].x),
                round(state["position"].y),
            ),
            state["radius"],
        )

        info = font.render(
            (
                f"FPS: {clock.get_fps():.0f} | "
                "WASD — движение | "
                "SPACE — цвет | ESC — выход"
            ),
            True,
            (50, 50, 50),
        )

        screen.blit(info, (20, 20))

        pygame.display.flip()

    # -------------------------------------------------
    # Главный цикл
    # -------------------------------------------------
    try:
        while state["running"]:

            dt = clock.tick(FPS) / 1000.0

            handle_events()

            direction = handle_input()

            update(direction, dt)

            render()

    finally:
        pygame.quit()


demo_with_functions()

# 6. Минимум, достаточный для самостоятельной работы

После этой демонстрации студенту достаточно знать следующие средства:

| Задача | PyGame |
|---|---|
| Инициализация | `pygame.init()` |
| Создание окна | `pygame.display.set_mode()` |
| Заголовок | `pygame.display.set_caption()` |
| Очередь событий | `pygame.event.get()` |
| Закрытие | `pygame.QUIT` |
| Однократное нажатие | `pygame.KEYDOWN` |
| Непрерывный ввод | `pygame.key.get_pressed()` |
| Частота кадров | `pygame.time.Clock()` |
| Вектор движения | `pygame.Vector2` |
| Прямоугольные области | `pygame.Rect` |
| Пересечение областей | `Rect.colliderect()` |
| Рисование круга | `pygame.draw.circle()` |
| Рисование прямоугольника | `pygame.draw.rect()` |
| Текст | `pygame.font.Font()` + `render()` |
| Вывод Surface | `screen.blit()` |
| Показ кадра | `pygame.display.flip()` |
| Завершение | `pygame.quit()` |

Этого достаточно, чтобы самостоятельно построить более содержательное приложение в рамках ПР № 2.
